# Cyber Hacking Breaches Prediction and Detection Using Machine Learning

Training notebook based on the supplied project notebook. The model uses URL text with TF-IDF and a Random Forest classifier.

In [ ]:
from pathlib import Path
import pickle
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
# Download upload.csv from the source noted in ../dataset/README.md.
project_root = Path.cwd()
if not (project_root / 'dataset').exists() and (project_root.parent / 'dataset').exists():
    project_root = project_root.parent
data_candidates = [project_root / 'dataset' / 'upload.csv', project_root / 'upload.csv']
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('upload.csv not found. Download it and place it in dataset/upload.csv; see dataset/README.md.')
data = pd.read_csv(data_path)
data.info()
data[['url', 'status']].head()


In [ ]:
X = data['url'].astype(str)
y = data['status']
print(y.value_counts())


In [ ]:
tfidf_vectorizer = TfidfVectorizer(min_df=1, stop_words='english')
X_tfidf = tfidf_vectorizer.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.3, random_state=250
)
print('Training shape:', X_train.shape)
print('Testing shape:', X_test.shape)


In [ ]:
rfc = RandomForestClassifier(random_state=0)
rfc.fit(X_train, y_train)
print('Training accuracy:', rfc.score(X_train, y_train))


In [ ]:
y_pred = rfc.predict(X_test)
print('Testing accuracy:', accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))


In [ ]:
sample_urls = ['https://jpinfotech.org/']
sample_vectors = tfidf_vectorizer.transform(sample_urls)
print('Prediction:', rfc.predict(sample_vectors)[0])


In [ ]:
# Save the trained model and vectorizer in the repository root for app.py.
with open(project_root / 'tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
with open(project_root / 'model.pkl', 'wb') as f:
    pickle.dump(rfc, f)
print('Saved model.pkl and tfidf_vectorizer.pkl to', project_root)
